In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

ANNOTATIONS = ROOT / "data" / "annotations"
RESULTS = ROOT / "results" / "vlm"
EVALUATION = ROOT / "results" / "evaluation"
EVALUATION.mkdir(parents=True, exist_ok=True)

human = pd.read_csv(
    ANNOTATIONS / "human_annotations.csv",
    keep_default_na=False
)

model = pd.read_csv(
    RESULTS / "vlm_predictions.csv"
)

FIELDS = [
    "greenery",
    "pedestrian",
    "cycling",
    "motor_vehicle"
]

comparison = human.merge(
    model[["image_id", *FIELDS]],
    on="image_id",
    suffixes=("_human", "_model"),
    validate="one_to_one"
)

assert len(comparison) == 40
assert comparison["location_id"].nunique() == 20

comparison.to_csv(
    EVALUATION / "human_vs_model.csv",
    index=False
)

display(comparison.head())

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

CLASSES = {
    "greenery": ["low", "medium", "high", "unclear"],
    "pedestrian": ["weak", "moderate", "strong", "unclear"],
    "cycling": ["absent", "present", "unclear"],
    "motor_vehicle": ["low", "medium", "high", "unclear"]
}

summary = []

for field in FIELDS:
    y_true = comparison[f"{field}_human"]
    y_pred = comparison[f"{field}_model"]

    labels = CLASSES[field]

    print(f"\n{'=' * 50}")
    print(field.upper())

    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        output_dict=True,
        zero_division=0
    )

    print(classification_report(
        y_true,
        y_pred,
        labels=labels,
        zero_division=0
    ))

    summary.append({
        "indicator": field,
        "accuracy": report["accuracy"],
        "macro_f1": report["macro avg"]["f1-score"]
    })

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=labels
    )

    fig, ax = plt.subplots(figsize=(6, 5))

    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=labels
    ).plot(ax=ax, cmap="Blues", colorbar=False)

    ax.set_title(f"{field.title()} — Human vs VLM")
    plt.tight_layout()

    fig.savefig(
        EVALUATION / f"confusion_{field}.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()

summary_df = pd.DataFrame(summary)
display(summary_df)

summary_df.to_csv(
    EVALUATION / "classification_summary.csv",
    index=False
)

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

location_ids = comparison["location_id"].unique()
rng.shuffle(location_ids)

# 8 locations for prompt development,
# 12 locations reserved for final evaluation.
dev_ids = set(location_ids[:8])

comparison["split"] = comparison["location_id"].apply(
    lambda x: "development" if x in dev_ids else "holdout"
)

print(comparison.groupby("split").agg(
    images=("image_id", "count"),
    locations=("location_id", "nunique")
))

comparison.to_csv(
    EVALUATION / "human_vs_model_with_split.csv",
    index=False
)

In [ ]:
dev = comparison[
    comparison["split"] == "development"
]

cycling_fp = dev[
    (dev["cycling_human"] == "absent") &
    (dev["cycling_model"] == "present")
]

print("Cycling false positives:", len(cycling_fp))

display(
    cycling_fp[
        ["image_id", "location_id",
         "cycling_human", "cycling_model"]
    ]
)

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed"
ANNOTATIONS = ROOT / "data" / "annotations"
VIEWS = PROCESSED / "perspective_views"
RESULTS = ROOT / "results" / "vlm"

RESULTS.mkdir(parents=True, exist_ok=True)

images = pd.read_csv(
    PROCESSED / "perspective_views_metadata.csv"
)

reference = pd.read_csv(
    ANNOTATIONS / "human_annotations.csv",
    keep_default_na=False
)

assert len(images) == 100
assert len(reference) == 40
assert set(reference["image_id"]).issubset(set(images["image_id"]))

print("Images for inference:", len(images))
print("Images with human labels:", len(reference))
print("All image files available:",
      all((VIEWS / f).exists() for f in images["filename"]))

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load the .env file from the project root
load_dotenv(ROOT / ".env")

# Verify that the key was loaded
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "API key not found. Check your .env file."
    )

client = OpenAI(api_key=api_key)

MODEL = "gpt-4.1-mini-2025-04-14"

print("API key loaded successfully.")
print("Model:", MODEL)

In [ ]:
import base64
from pydantic import BaseModel
from IPython.display import display
from PIL import Image

class VisualEvidence(BaseModel):
    greenery: str
    pedestrian: str
    cycling: str
    motor_vehicle: str

EVIDENCE_PROMPT = """
Examine this streetscape image.

For each of the following indicators, describe the
specific visible objects or features that provide evidence:

1. Greenery: trees, shrubs, grass or other vegetation.
2. Pedestrian infrastructure: sidewalks, pedestrian
   paths, crossings and pedestrianized areas.
3. Cycling infrastructure: designated cycle lanes,
   cycle tracks or other clearly designated facilities.
4. Motor-vehicle infrastructure: carriageways,
   parking spaces and other vehicle-oriented space.

IMPORTANT:
A bicycle, cyclist or parked bicycle does not establish
the presence of dedicated cycling infrastructure.
Clearly distinguish these objects from actual facilities.

Describe only what is visible. Do not invent features.
If evidence is ambiguous, say so.

Do not assign categorical labels.
"""

def explain_image(image_path):
    encoded = base64.b64encode(
        image_path.read_bytes()
    ).decode("utf-8")

    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": EVIDENCE_PROMPT
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "Describe the visual evidence."
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{encoded}",
                            "detail": "high"
                        }
                    }
                ]
            }
        ],
        response_format=VisualEvidence,
        temperature=0
    )

    return response.choices[0].message.parsed

In [ ]:
image_id = cycling_fp.iloc[5]["image_id"]

filename = images.loc[
    images["image_id"] == image_id,
    "filename"
].iloc[0]

image_path = VIEWS / filename

display(Image.open(image_path))

evidence = explain_image(image_path)

print("IMAGE:", image_id)

for indicator, description in evidence.model_dump().items():
    print(f"\n{indicator.upper()}:")
    print(description)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import pandas as pd

revised = pd.read_csv(
    RESULTS / "evidence_first_development.csv"
)

dev = comparison[
    comparison["split"] == "development"
].copy()

dev = dev.merge(
    revised[["image_id", *FIELDS]],
    on="image_id",
    suffixes=("", "_revised"),
    validate="one_to_one"
)

assert len(dev) == 16

rows = []

for field in FIELDS:
    human = dev[f"{field}_human"]
    original = dev[f"{field}_model"]
    updated = dev[field]

    # Exclude unsupported classes from macro-F1,
    # while retaining all predictions and errors.
    supported = sorted(human.unique())

    for name, predicted in [
        ("Original", original),
        ("Evidence-first", updated)
    ]:
        rows.append({
            "indicator": field,
            "prompt": name,
            "accuracy": accuracy_score(human, predicted),
            "macro_f1": f1_score(
                human,
                predicted,
                labels=supported,
                average="macro",
                zero_division=0
            )
        })

results = pd.DataFrame(rows)

display(
    results.pivot(
        index="indicator",
        columns="prompt",
        values=["accuracy", "macro_f1"]
    ).round(3)
)

results.to_csv(
    EVALUATION / "development_comparison.csv",
    index=False
)

In [ ]:
labels = ["absent", "present", "unclear"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, column, title in zip(
    axes,
    ["cycling_model", "cycling"],
    ["Original prompt", "Evidence-first prompt"]
):
    cm = confusion_matrix(
        dev["cycling_human"],
        dev[column],
        labels=labels
    )

    ConfusionMatrixDisplay(
        cm,
        display_labels=labels
    ).plot(
        ax=ax,
        cmap="Blues",
        colorbar=False
    )

    ax.set_title(title)

plt.tight_layout()
plt.show()

In [ ]:
reviewed = pd.read_csv(
    ANNOTATIONS / "human_annotations_reviewed.csv"
)

revised = pd.read_csv(
    RESULTS / "evidence_first_development.csv"
)

evaluation = reviewed.merge(
    model[["image_id", *FIELDS]],
    on="image_id",
    validate="one_to_one"
).merge(
    revised[["image_id", *FIELDS]],
    on="image_id",
    suffixes=("_original_model", "_evidence_first"),
    validate="one_to_one"
)

# Use the reviewed labels where available.
# Cycling and motor-vehicle labels remain unchanged.
for field in ["greenery", "pedestrian"]:
    evaluation[f"human_{field}"] = evaluation[
        f"reviewed_{field}"
    ]

for field in ["cycling", "motor_vehicle"]:
    evaluation[f"human_{field}"] = evaluation[
        f"original_{field}"
    ]

from sklearn.metrics import accuracy_score, f1_score

results = []

for field in FIELDS:
    truth = evaluation[f"human_{field}"]
    supported = sorted(truth.unique())

    for name, column in [
        ("Original", f"{field}_original_model"),
        ("Evidence-first", f"{field}_evidence_first")
    ]:
        prediction = evaluation[column]

        results.append({
            "indicator": field,
            "prompt": name,
            "accuracy": accuracy_score(truth, prediction),
            "macro_f1": f1_score(
                truth,
                prediction,
                labels=supported,
                average="macro",
                zero_division=0
            )
        })

updated_results = pd.DataFrame(results)

display(
    updated_results.pivot(
        index="indicator",
        columns="prompt",
        values=["accuracy", "macro_f1"]
    ).round(3)
)

evaluation.to_csv(
    EVALUATION / "reviewed_development_comparison.csv",
    index=False
)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

hybrid = evaluation.copy()

strategy = {
    "greenery": "original_model",
    "pedestrian": "original_model",
    "cycling": "evidence_first",
    "motor_vehicle": "original_model"
}

rows = []

for field, source in strategy.items():
    truth = hybrid[f"human_{field}"]
    predicted = hybrid[f"{field}_{source}"]

    rows.append({
        "indicator": field,
        "accuracy": accuracy_score(truth, predicted),
        "macro_f1": f1_score(
            truth,
            predicted,
            labels=sorted(truth.unique()),
            average="macro",
            zero_division=0
        )
    })

hybrid_results = pd.DataFrame(rows)
display(hybrid_results.round(3))

hybrid.to_csv(
    EVALUATION / "hybrid_development.csv",
    index=False
)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

holdout = pd.read_csv(
    ANNOTATIONS / "human_annotations_holdout_reviewed.csv"
)

original = pd.read_csv(
    RESULTS / "vlm_predictions.csv"
)

evidence = pd.read_csv(
    RESULTS / "evidence_first_holdout.csv"
)

final = (
    holdout
    .merge(
        original[["image_id", *FIELDS]],
        on="image_id",
        validate="one_to_one"
    )
    .merge(
        evidence[["image_id", "cycling"]]
        .rename(columns={"cycling": "cycling_evidence"}),
        on="image_id",
        validate="one_to_one"
    )
)

assert len(final) == 24

strategy = {
    "greenery": ("reviewed_greenery", "greenery"),
    "pedestrian": ("reviewed_pedestrian", "pedestrian"),
    "cycling": ("original_cycling", "cycling_evidence"),
    "motor_vehicle": ("original_motor_vehicle", "motor_vehicle")
}

rows = []

for field, (truth_col, prediction_col) in strategy.items():
    truth = final[truth_col]
    predicted = final[prediction_col]

    rows.append({
        "indicator": field,
        "accuracy": accuracy_score(truth, predicted),
        "macro_f1": f1_score(
            truth,
            predicted,
            labels=sorted(truth.unique()),
            average="macro",
            zero_division=0
        )
    })

final_results = pd.DataFrame(rows)

display(final_results.round(3))

final_results.to_csv(
    EVALUATION / "final_holdout_metrics.csv",
    index=False
)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

comparison_rows = []

for field, (truth_col, hybrid_col) in strategy.items():
    truth = final[truth_col]
    original_pred = final[field]
    hybrid_pred = final[hybrid_col]

    for method, prediction in [
        ("Original", original_pred),
        ("Hybrid", hybrid_pred)
    ]:
        comparison_rows.append({
            "indicator": field,
            "method": method,
            "accuracy": accuracy_score(truth, prediction),
            "macro_f1": f1_score(
                truth,
                prediction,
                labels=sorted(truth.unique()),
                average="macro",
                zero_division=0
            )
        })

holdout_comparison = pd.DataFrame(comparison_rows)

display(
    holdout_comparison.pivot(
        index="indicator",
        columns="method",
        values=["accuracy", "macro_f1"]
    ).round(3)
)

holdout_comparison.to_csv(
    EVALUATION / "original_vs_hybrid_holdout.csv",
    index=False
)